# GRPO-LoRA on Free Colab — ISOPro Comparison Notebook

Runs **vanilla GRPO with LoRA on a 4-bit quantized base** for the GCE paper's ISOPro vs. GRPO comparison. One notebook handles all six configurations from Section 7:

| Model              | Scheduling | MBPP |
|--------------------|:----------:|:----:|
| Qwen 2.5 3B        | ✓          | ✓    |
| Llama 3.2 3B       | ✓          | ✓    |
| Gemma 2 2B/4B      | ✓          | ✓    |

**Hardware target**: free-tier Colab T4 (16 GB VRAM). Backbone: Unsloth's GRPO patch (memory-efficient reference logprobs, no separate frozen reference model in VRAM) + TRL `GRPOTrainer`.

**Output**: `experiment_results.json` in the same schema as `examples/run_scheduling_experiment.py`, so `examples/watch_curriculum_emerge.py` and the existing analysis tooling work on this output unchanged.

**One config to change**: edit the cell labeled **CONFIG** below, then `Runtime → Run all`. Re-run with a different `MODEL_ID` / `DOMAIN` to fill in another cell of the table.

## 1. GPU + dependency check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Expect a T4 with 15360 MiB, or A100/L4 if you have Pro.

In [ ]:
%%capture
# Unsloth pin matches the version that ships GRPO patches at time of writing.
# `vllm` powers the high-throughput rollout step inside GRPOTrainer.
!pip install -q --upgrade pip
!pip install -q unsloth vllm
!pip install -q --upgrade --no-deps trl peft accelerate bitsandbytes datasets
!pip install -q ortools  # OR-Tools is required for the scheduling task bank.

## 2. CONFIG — edit this cell

Pick one of `MODEL_ID` and one of `DOMAIN`, then run all cells. Each (model, domain) pair takes ~30–60 min on T4.

In [ ]:
# ============================================================================
#                                  CONFIG
# ============================================================================

MODEL_ID = "unsloth/Qwen2.5-3B-Instruct"
# Alternatives:
#   "unsloth/Llama-3.2-3B-Instruct"
#   "unsloth/gemma-2-2b-it"           # safe on free T4
#   "unsloth/gemma-2-9b-it"           # needs A100 / L4 (Pro)

DOMAIN = "scheduling"   # "scheduling" | "mbpp"

# Branch of iso-ai/isopro to install (must contain the latest task bank code).
ISOPRO_BRANCH = "public"

# Training schedule — chosen to mirror the scheduling experiment in the paper
# (6 iterations, ~84 rollouts/iter) while staying inside T4 memory.
N_ITERATIONS         = 6
STEPS_PER_ITERATION  = 14         # one "iteration" = N gradient steps
GROUP_SIZE           = 4          # G in GRPO (=k_samples in ISOPro)
TASKS_PER_ITERATION  = 21         # G * tasks_per_iter ≈ 84 rollouts/iter
MAX_PROMPT_LENGTH    = 768
MAX_COMPLETION_LEN   = 384
LEARNING_RATE        = 5e-6
LORA_RANK            = 16
LORA_ALPHA           = 16
EVAL_PER_TIER        = 5          # held-out problems per tier
SEED                 = 42

# Where to save the run. Mount Drive in the next cell if you want persistence.
OUTPUT_DIR  = "/content/grpo_lora_run"
RESULTS_FILE = f"{OUTPUT_DIR}/experiment_results.json"

In [ ]:
# Optional: persist results to your Google Drive so reruns survive disconnects.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/grpo_lora_runs/" + MODEL_ID.split("/")[-1] + "_" + DOMAIN
    RESULTS_FILE = f"{OUTPUT_DIR}/experiment_results.json"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Install ISOPro and import task banks

We pull the task generators and verifiers directly from the repo so the GRPO run uses the *same* problem distribution and the *same* deterministic verifier that the ISOPro runs use. Apples-to-apples.

In [ ]:
%%capture
!pip install -q "git+https://github.com/iso-ai/isopro.git@{ISOPRO_BRANCH}"

In [ ]:
# Wire up the right task bank + verifier for the chosen domain.
if DOMAIN == "scheduling":
    from isopro.environments.tasks.scheduling_tasks import (
        SchedulingTier, build_tier_task_bank, build_eval_set,
    )
    from isopro.environments.tasks.scheduling_verifier import score_scheduling_task as score_fn

    # Tier 5 (full composition) is the held-out tier (matches paper).
    train_pool = {
        tier: build_tier_task_bank(tier, n_problems=80, seed=SEED)
        for tier in SchedulingTier if tier != SchedulingTier.FULL_COMPOSITION
    }
    eval_set = build_eval_set(n_per_tier=EVAL_PER_TIER, seed=SEED + 1)
    TIER_VALUES = [t.value for t in SchedulingTier]

elif DOMAIN == "mbpp":
    from isopro.environments.tasks.mbpp_tasks import (
        MBPPTier, build_mbpp_splits, generate_mbpp_task,
    )
    from isopro.environments.tasks.mbpp_verifier import score_mbpp_task as score_fn

    splits = build_mbpp_splits(eval_per_tier=EVAL_PER_TIER, seed=SEED)
    train_pool = {
        tier: [generate_mbpp_task(p) for p in splits["train"][tier]]
        for tier in MBPPTier if tier != MBPPTier.HELD_OUT
    }
    eval_set = {
        tier.value: [generate_mbpp_task(p) for p in splits["eval"][tier]]
        for tier in MBPPTier
    }
    TIER_VALUES = [t.value for t in MBPPTier]

else:
    raise ValueError(f"Unknown domain: {DOMAIN}")

print(f"Loaded {sum(len(v) for v in train_pool.values())} training tasks across "
      f"{len(train_pool)} tiers, with {sum(len(v) for v in eval_set.values())} held-out eval tasks.")

## 4. Build the HuggingFace Dataset and the reward function

Each row carries the prompt + the original task ID so the reward function can look up the verifier inputs. The reward function returns binary {0.0, 1.0} from the *same* deterministic verifier used by ISOPro.

In [ ]:
import random
from datasets import Dataset

rng = random.Random(SEED)

# Flatten train_pool to a list, but keep the tier label for per-tier accounting.
training_tasks = []
for tier, tasks in train_pool.items():
    for t in tasks:
        training_tasks.append((tier.value if hasattr(tier, "value") else tier, t))
rng.shuffle(training_tasks)

# Lookup table: task_id → (Task, tier_label) so the reward func can score.
task_lookup = {t.task_id: (t, tier) for tier, t in training_tasks}

ds_rows = [
    {
        "prompt": [{"role": "user", "content": t.prompt}],
        "task_id": t.task_id,
        "tier": tier,
    }
    for tier, t in training_tasks
]
train_dataset = Dataset.from_list(ds_rows)
print(f"Training dataset: {len(train_dataset)} rows")

In [ ]:
def correctness_reward(completions, task_id, **_):
    """Verifier-grounded reward: 1.0 iff the generated text passes verification.

    Args:
        completions: list of conversation-format completions from TRL.
        task_id:    list of task IDs from the dataset row.

    Returns:
        list[float]: binary reward per generation.
    """
    rewards = []
    for completion, tid in zip(completions, task_id):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        task, _ = task_lookup[tid]
        score, _detail = score_fn(task, text)
        rewards.append(float(score))
    return rewards

## 5. Load the model with Unsloth (4-bit + LoRA)

Unsloth's `PatchFastRL("GRPO", ...)` rewrites the reference-logprob computation so we don't need a separate frozen reference model in VRAM — the policy logprobs are computed by toggling LoRA off in the same forward pass. This is the architectural concession that lets vanilla GRPO run on a single T4.

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name        = MODEL_ID,
    max_seq_length    = MAX_PROMPT_LENGTH + MAX_COMPLETION_LEN,
    load_in_4bit      = True,
    fast_inference    = True,         # vLLM-backed rollout generation
    max_lora_rank     = LORA_RANK,
    gpu_memory_utilization = 0.5,     # leave room for vLLM KV cache
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state   = SEED,
)

## 6. Per-iteration evaluation callback

TRL's `GRPOTrainer` doesn't expose an iteration boundary natively, so we use a `TrainerCallback` that fires every `STEPS_PER_ITERATION` steps, runs the held-out eval set through the (currently LoRA-active) model, and appends one row to `history` in the same schema as `examples/run_scheduling_experiment.py`.

In [ ]:
import json
import time
from collections import Counter
from transformers import TrainerCallback
from vllm import SamplingParams

history: list[dict] = []
rollout_log: list[dict] = []          # (step, task_id, tier, score) for buffer composition
experiment_start = time.time()

EVAL_SAMPLING = SamplingParams(
    temperature       = 0.1,
    top_p             = 0.95,
    max_tokens        = MAX_COMPLETION_LEN,
)

def _vllm_generate(model, tokenizer, prompts, sampling_params):
    """Run vLLM under the LoRA-active model. Returns list[str] completions."""
    chat_prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=False, add_generation_prompt=True,
        )
        for p in prompts
    ]
    outs = model.fast_generate(chat_prompts, sampling_params=sampling_params,
                               lora_request=model.load_lora("grpo_saved_lora"))
    return [o.outputs[0].text for o in outs]

def evaluate_per_tier():
    """Score the held-out eval set and return tier_label -> accuracy."""
    scores: dict[str, list[float]] = {tier: [] for tier in TIER_VALUES}
    flat_prompts, flat_keys = [], []
    for tier_label, tasks in eval_set.items():
        # tier_label may be MBPPTier/SchedulingTier obj or str — normalise.
        tier_label = tier_label.value if hasattr(tier_label, "value") else tier_label
        for t in tasks:
            flat_prompts.append(t.prompt)
            flat_keys.append((tier_label, t))

    completions = _vllm_generate(model, tokenizer, flat_prompts, EVAL_SAMPLING)
    for (tier_label, task), text in zip(flat_keys, completions):
        score, _ = score_fn(task, text)
        scores[tier_label].append(score)

    return {k: (sum(v) / len(v) if v else 0.0) for k, v in scores.items()}

class IterationCallback(TrainerCallback):
    """Run held-out eval every STEPS_PER_ITERATION training steps."""
    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step == 0 or step % STEPS_PER_ITERATION != 0:
            return
        iteration = step // STEPS_PER_ITERATION

        eval_scores = evaluate_per_tier()

        # Buffer-composition analogue: count correct rollouts seen so far per tier.
        buffer_comp = Counter(r["tier"] for r in rollout_log if r["score"] >= 1.0)
        n_rollouts = len(rollout_log)
        n_correct  = sum(1 for r in rollout_log if r["score"] >= 1.0)

        last_loss = state.log_history[-1].get("loss") if state.log_history else None
        wall_clock = time.time() - experiment_start

        history.append({
            "iteration":          iteration,
            "tier_accuracies":    eval_scores,
            "buffer_composition": dict(buffer_comp),
            "n_rollouts":         n_rollouts,
            "n_correct":          n_correct,
            "train_loss":         last_loss,
            "wall_clock_s":       round(wall_clock, 2),
            "peak_memory_mb":     None,   # filled in at save time from torch
        })
        # Snapshot the partial JSON each iteration in case of disconnect.
        save_results()
        print(f"[iter {iteration}/{N_ITERATIONS}] eval: {eval_scores} "
              f"buffer: {dict(buffer_comp)} loss: {last_loss}")

def save_results():
    import torch
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else None
    if history and peak_mb is not None:
        history[-1]["peak_memory_mb"] = round(peak_mb, 1)
    payload = {
        "experiment": {
            "method":      "grpo_lora",
            "model_id":    MODEL_ID,
            "domain":      DOMAIN,
            "seed":        SEED,
            "n_iterations":N_ITERATIONS,
            "steps_per_iteration": STEPS_PER_ITERATION,
            "group_size":  GROUP_SIZE,
            "learning_rate": LEARNING_RATE,
            "lora_rank":   LORA_RANK,
            "hardware":    "colab_t4",
        },
        "isopro_loop": {  # named for backwards compat with watch_curriculum_emerge.py
            "history": history,
        },
        "total_wall_clock_s": round(time.time() - experiment_start, 2),
    }
    with open(RESULTS_FILE, "w") as f:
        json.dump(payload, f, indent=2)

class RolloutLoggingReward:
    """Wraps the verifier reward fn to log every rollout for buffer composition."""
    __name__ = "correctness_reward"
    def __call__(self, completions, task_id, tier, **_):
        rewards = []
        for completion, tid, ti in zip(completions, task_id, tier):
            text = completion[0]["content"] if isinstance(completion, list) else str(completion)
            task, _ = task_lookup[tid]
            score, _ = score_fn(task, text)
            rollout_log.append({"task_id": tid, "tier": ti, "score": float(score)})
            rewards.append(float(score))
        return rewards

logging_reward = RolloutLoggingReward()

## 7. Configure GRPOTrainer and train

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    use_vllm                     = True,
    learning_rate                = LEARNING_RATE,
    adam_beta1                   = 0.9,
    adam_beta2                   = 0.99,
    weight_decay                 = 0.1,
    warmup_ratio                 = 0.1,
    lr_scheduler_type            = "cosine",
    optim                        = "paged_adamw_8bit",
    logging_steps                = 1,
    bf16                         = True,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 1,
    num_generations              = GROUP_SIZE,
    max_prompt_length            = MAX_PROMPT_LENGTH,
    max_completion_length        = MAX_COMPLETION_LEN,
    max_steps                    = N_ITERATIONS * STEPS_PER_ITERATION,
    save_steps                   = N_ITERATIONS * STEPS_PER_ITERATION,
    output_dir                   = OUTPUT_DIR,
    seed                         = SEED,
    report_to                    = "none",
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [logging_reward],
    args             = training_args,
    train_dataset    = train_dataset,
    callbacks        = [IterationCallback()],
)
trainer.train()

## 8. Final save + sanity check

After training completes, we run one last eval and write the final JSON. The output file matches the schema produced by `examples/run_scheduling_experiment.py`, so:

```bash
python examples/watch_curriculum_emerge.py --log /content/grpo_lora_run/experiment_results.json
```

...just works.

In [ ]:
save_results()
print(f"Wrote {RESULTS_FILE}")
print("\n--- Final history (last iteration) ---")
print(json.dumps(history[-1] if history else {}, indent=2))

In [ ]:
# Quick visualisation right inside the notebook so you can sanity-check before downloading.
import matplotlib.pyplot as plt

iters    = [h["iteration"] for h in history]
by_tier  = {tier: [h["tier_accuracies"].get(tier, 0.0) for h in history] for tier in TIER_VALUES}

fig, ax = plt.subplots(figsize=(8, 4))
for tier, accs in by_tier.items():
    ax.plot(iters, accs, marker="o", label=tier)
ax.set_xlabel("iteration"); ax.set_ylabel("per-tier accuracy")
ax.set_title(f"GRPO-LoRA on {MODEL_ID.split('/')[-1]} — {DOMAIN}")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

## 9. Pulling results back into the repo

Download `experiment_results.json` and place it at:

```
data/grpo_lora_<model>_<domain>/experiment_results.json
```

Then on your laptop:

```bash
python examples/watch_curriculum_emerge.py \
    --log data/grpo_lora_qwen25_3b_scheduling/experiment_results.json
```

Side-by-side with `data/scheduling_experiment_qwen/experiment_results.json` (the ISOPro run) for the paper's Section 7 comparison.